# Integrated Taxi Trips

This notebook creates `data/processed/integrated_taxi_trips`, a Delta table with one row per valid taxi trip enriched with pickup/dropoff zone details, hourly weather, and hourly PM2.5 context.

## Integration strategy

- **Zone context:** join `pickup_location_id` and `dropoff_location_id` to the taxi-zone lookup. Both are left joins, so an unknown zone never removes a trip.
- **Weather:** `pickup_hour` is already the pickup timestamp truncated to the hour. It is matched exactly to the weather `observation_timestamp`. The weather data is treated as the representative NYC hourly observation supplied for this project.
- **Air quality:** the source contains hourly PM2.5 observations (`parameter_code = 88101`) from multiple monitors. New York county codes are mapped to taxi boroughs, then measurements are aggregated by borough and local observation hour. Each trip receives the mean/min/max PM2.5 value and monitor counts for its pickup borough and hour.
- **Missing context:** all joins are left joins. Missing weather, air-quality, or zone context remains `null`; match-status columns make missingness explicit. Values are not fabricated or replaced with zero.

## Limitations

- This run preserved all 9,417,689 taxi trips and matched both pickup and dropoff zones at 100%. These rates only describe the current input files; future trips with unknown or newly introduced location IDs will retain null zone context.
- Weather matched 100% of trips by pickup hour in this run, but a matched weather row can still contain null measurements (for example, missing precipitation). Match status therefore indicates temporal coverage, not completeness of every weather field.
- Air-quality coverage is only 9.95% with the strict borough-and-exact-hour join, leaving 90.05% of trips without PM2.5 context. Missing PM2.5 must not be interpreted or imputed as zero.
- Trips only contain zone IDs, not pickup coordinates, so air quality cannot be assigned to the nearest monitor. A borough-hour average is a coarse spatial approximation and can hide within-borough variation.
- Exact hourly matching does not interpolate or carry observations across time. This avoids fabricating measurements but makes results sensitive to the sparse monitoring schedule.
- A single weather series cannot capture microclimate differences across NYC boroughs.
- The timestamps are interpreted in the Spark session timezone `America/New_York`; daylight-saving transitions may introduce ambiguous local hours.
- Context is based on pickup time only and does not describe conditions during the whole trip.
- Taxi ingestion accepted 9,417,689 of 9,554,778 source rows and rejected 137,089 rows (1.43%). The integrated table deliberately contains only accepted trips, so its 9,417,689-row count must not be interpreted as the complete raw taxi population.
- The generated `trip_id` is a hash of selected trip attributes rather than a source-issued identifier. A correction to any hashed attribute changes the ID, while otherwise identical source rows receive the same ID; one duplicate was detected in this run.
- Taxi ingestion took approximately 968.5 seconds in this run. The framework performs several full-data Spark actions for auditing (`input`, accepted, rejected, and duplicate counts), computes hashes and duplicate windows, writes separate processed and failed Delta tables, and appends run metadata. Consequently, ingestion time is substantially greater than a simple Parquet read.
- Ingestion and integration are full-refresh operations. Re-running them overwrites and recomputes the complete taxi output instead of processing only new or changed source partitions.
- The current local Windows configuration (`local[4]`, 3 GB JVM heap) prioritizes stability over throughput. CPU parallelism and in-memory caching are limited, so large shuffles and validation intermediates spill to D-disk; runtime and temporary storage increase with input size.
- Even small datasets have fixed Spark scheduling and Delta transaction overhead: this run took about 30.9 seconds for 265 taxi-zone rows and 18.4 seconds for 8,784 weather rows. Elapsed time therefore does not scale only with row count.

In [ ]:
from pathlib import Path

from data_intensive_computing.spark_env import configure_windows_spark_env

configure_windows_spark_env()

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "integrated_taxi_trips"

builder = (
    SparkSession.builder
    .appName("Integrated-Taxi-Trips")
    .master("local[4]")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.local.dir", __import__("os").environ["SPARK_LOCAL_DIRS"])
    .config("spark.driver.extraJavaOptions", f'-Djava.io.tmpdir={__import__("os").environ["SPARK_JAVA_TMP_DIR"]}')
    .config("spark.jars.ivy", __import__("os").environ["SPARK_IVY_DIR"])
    .config("spark.sql.warehouse.dir", __import__("os").environ["SPARK_WAREHOUSE_DIR"])
    .config("spark.hadoop.hadoop.tmp.dir", __import__("os").environ["HADOOP_TMP_DIR"])
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", __import__("os").environ["PYSPARK_PYTHON"])
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Processed data:", PROCESSED_DIR)
print("Output Delta table:", OUTPUT_PATH)

In [ ]:
INPUT_PATHS = {
    "taxi_trips": PROCESSED_DIR / "taxi_trips",
    "weather": PROCESSED_DIR / "weather",
    "air_quality": PROCESSED_DIR / "air_quality",
    "taxi_zones": PROCESSED_DIR / "taxi_zones",
}

missing_paths = [name for name, path in INPUT_PATHS.items() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing processed Delta tables: " + ", ".join(missing_paths)
        + ". Run ingestion_framework.ipynb first."
    )

trips = spark.read.format("delta").load(str(INPUT_PATHS["taxi_trips"]))
weather = spark.read.format("delta").load(str(INPUT_PATHS["weather"]))
air_quality = spark.read.format("delta").load(str(INPUT_PATHS["air_quality"]))
zones = spark.read.format("delta").load(str(INPUT_PATHS["taxi_zones"]))

required_columns = {
    "taxi_trips": {"trip_id", "pickup_hour", "pickup_location_id", "dropoff_location_id"},
    "weather": {"observation_timestamp", "temp", "rhum", "prcp", "wspd"},
    "air_quality": {
        "observation_timestamp", "state_code", "county_code", "parameter_code",
        "sample_measurement", "station_id", "units_of_measure",
    },
    "taxi_zones": {"location_id", "zone", "borough"},
}
for name, df in {
    "taxi_trips": trips,
    "weather": weather,
    "air_quality": air_quality,
    "taxi_zones": zones,
}.items():
    missing = required_columns[name] - set(df.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {sorted(missing)}")
    print(f"{name}: {len(df.columns)} columns")

In [ ]:
# Add pickup and dropoff geography. The lookup is small, so broadcast joins avoid shuffling trips.
pickup_zones = F.broadcast(
    zones.select(
        F.col("location_id").alias("pickup_zone_location_id"),
        F.col("zone").alias("pickup_zone"),
        F.col("borough").alias("pickup_borough"),
    )
)
dropoff_zones = F.broadcast(
    zones.select(
        F.col("location_id").alias("dropoff_zone_location_id"),
        F.col("zone").alias("dropoff_zone"),
        F.col("borough").alias("dropoff_borough"),
    )
)

trips_with_zones = (
    trips
    .join(
        pickup_zones,
        trips.pickup_location_id == pickup_zones.pickup_zone_location_id,
        "left",
    )
    .drop("pickup_zone_location_id")
    .join(
        dropoff_zones,
        trips.dropoff_location_id == dropoff_zones.dropoff_zone_location_id,
        "left",
    )
    .drop("dropoff_zone_location_id")
)

In [ ]:
# Weather is one representative NYC row per hour; match it exactly to pickup_hour.
weather_hourly = F.broadcast(
    weather.select(
        F.col("observation_timestamp").alias("weather_observation_timestamp"),
        F.col("temp").alias("weather_temperature_c"),
        F.col("rhum").alias("weather_relative_humidity_pct"),
        F.col("prcp").alias("weather_precipitation_mm"),
        F.col("snwd").alias("weather_snow_depth_mm"),
        F.col("wdir").alias("weather_wind_direction_deg"),
        F.col("wspd").alias("weather_wind_speed_kmh"),
        F.col("wpgt").alias("weather_wind_gust_kmh"),
        F.col("pres").alias("weather_pressure_hpa"),
        F.col("cldc").alias("weather_cloud_cover_code"),
        F.col("coco").alias("weather_condition_code"),
    )
)

trips_with_weather = (
    trips_with_zones
    .join(
        weather_hourly,
        trips_with_zones.pickup_hour == weather_hourly.weather_observation_timestamp,
        "left",
    )
    .withColumn(
        "weather_match_status",
        F.when(F.col("weather_observation_timestamp").isNotNull(), F.lit("exact_hour"))
         .otherwise(F.lit("missing")),
    )
)

In [ ]:
# NYC boroughs correspond to New York State county codes in the air-quality data.
# Using codes is more stable than matching display names.
nyc_counties = spark.createDataFrame(
    [
        ("005", "Bronx"),
        ("047", "Brooklyn"),
        ("061", "Manhattan"),
        ("081", "Queens"),
        ("085", "Staten Island"),
    ],
    ["county_code", "air_quality_borough"],
)

nyc_air_quality = (
    air_quality
    .filter((F.col("state_code") == "36") & (F.col("parameter_code") == "88101"))
    .join(F.broadcast(nyc_counties), "county_code", "inner")
)

# Multiple monitors/methods can report within the same borough-hour.
# Aggregate first so the subsequent trip join remains many-to-one.
air_quality_hourly = (
    nyc_air_quality
    .groupBy("air_quality_borough", "observation_timestamp")
    .agg(
        F.avg("sample_measurement").alias("air_quality_pm25_mean"),
        F.min("sample_measurement").alias("air_quality_pm25_min"),
        F.max("sample_measurement").alias("air_quality_pm25_max"),
        F.count("sample_measurement").alias("air_quality_observation_count"),
        F.countDistinct("station_id").alias("air_quality_station_count"),
        F.first("units_of_measure", ignorenulls=True).alias("air_quality_units"),
    )
    .withColumnRenamed("observation_timestamp", "air_quality_observation_timestamp")
)

trips_integrated = (
    trips_with_weather
    .join(
        F.broadcast(air_quality_hourly),
        (trips_with_weather.pickup_borough == air_quality_hourly.air_quality_borough)
        & (trips_with_weather.pickup_hour == air_quality_hourly.air_quality_observation_timestamp),
        "left",
    )
    .drop("air_quality_borough")
    .withColumn(
        "air_quality_match_status",
        F.when(F.col("air_quality_observation_timestamp").isNotNull(), F.lit("borough_exact_hour"))
         .otherwise(F.lit("missing")),
    )
    .withColumn("_integration_strategy_version", F.lit("1.0.0"))
    .withColumn("_integrated_at", F.current_timestamp())
)

## Write the integrated Delta table

The joins above are lazy. The following cell performs the full computation and is therefore the long-running step. The output is partitioned consistently with the taxi-trip source by pickup year and month.

In [ ]:
(
    trips_integrated.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("pickup_year", "pickup_month")
    .save(str(OUTPUT_PATH))
)

print("Integrated Delta table written to:", OUTPUT_PATH)

In [ ]:
integrated = spark.read.format("delta").load(str(OUTPUT_PATH))

input_rows = trips.count()
quality = integrated.agg(
    F.count("*").alias("integrated_rows"),
    F.avg((F.col("pickup_zone").isNotNull()).cast("double")).alias("pickup_zone_match_rate"),
    F.avg((F.col("dropoff_zone").isNotNull()).cast("double")).alias("dropoff_zone_match_rate"),
    F.avg((F.col("weather_match_status") == "exact_hour").cast("double")).alias("weather_match_rate"),
    F.avg((F.col("air_quality_match_status") == "borough_exact_hour").cast("double")).alias("air_quality_match_rate"),
).first()

assert quality["integrated_rows"] == input_rows, (
    f"Trip cardinality changed: input={input_rows}, output={quality['integrated_rows']}"
)

print(f"Input/output rows: {input_rows:,}")
for metric in [
    "pickup_zone_match_rate",
    "dropoff_zone_match_rate",
    "weather_match_rate",
    "air_quality_match_rate",
]:
    print(f"{metric}: {quality[metric]:.2%}")

integrated.select(
    "trip_id",
    "pickup_datetime",
    "pickup_zone",
    "pickup_borough",
    "dropoff_zone",
    "dropoff_borough",
    "weather_temperature_c",
    "weather_precipitation_mm",
    "air_quality_pm25_mean",
    "air_quality_station_count",
    "weather_match_status",
    "air_quality_match_status",
).show(10, truncate=False)

## Result

`integrated_taxi_trips` preserves the taxi trip as the table grain and primary analytical entity. Context columns are nullable by design. Consumers can filter on `weather_match_status` and `air_quality_match_status` when an analysis requires complete context, while analyses that do not need those measurements can retain every valid trip.

The output can be reopened in another notebook with:

```python
integrated = spark.read.format("delta").load(
    str(PROJECT_ROOT / "data" / "processed" / "integrated_taxi_trips")
)
```